# H-T01~H-T04 E300 continuation

기존 `checkpoint_final.pth`의 network, optimizer, epoch, scheduler 위치를 이어받아 별도 결과 폴더에서 정확히 300 epoch까지 학습합니다. Early stopping은 사용하지 않으며 5 epoch마다 Full-volume Macro Dice를 계산합니다.


In [ ]:
from pathlib import Path
import json, os, subprocess, time
import pandas as pd
from IPython.display import clear_output, display

ROOT=Path("/home/stu03/workspace/model")
DATA_ROOT=Path("/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet")
PYTHON=ROOT/".venv/bin/python"
BATCH_DIR=ROOT/"runs/batches/H-T01_H-T04_E300"
BATCH_STATUS=BATCH_DIR/"batch_status.json"
assert PYTHON.is_file() and DATA_ROOT.is_dir()


## 1. 설정·부모 checkpoint·plan 확인

학습은 시작하지 않습니다. 네 부모 checkpoint epoch와 새 plan을 검사합니다.


In [ ]:
subprocess.run([str(PYTHON),"-m","pytest","-q","tests/test_h_tuning_e300_continuation.py"],cwd=ROOT,check=True)
subprocess.run([str(PYTHON),"scripts/preprocessing/prepare_h_t01_h_t04_e300_plans.py","--preprocessed-root",str(DATA_ROOT/"nnUNet_preprocessed")],cwd=ROOT,check=True)
subprocess.run([str(PYTHON),"scripts/training/run_h_t01_h_t04_e300.py","--data-root",str(DATA_ROOT),"--dry-run"],cwd=ROOT,check=True)


## 2. 백그라운드 순차 실행

최초 실행은 `RESUME=False`입니다. E300 실행이 중단된 뒤 이어갈 때만 `True`로 바꾸세요. 완료된 E300 실험은 자동으로 건너뜁니다.


In [ ]:
RESUME=False
GPU="0"
BATCH_DIR.mkdir(parents=True,exist_ok=True)
command=[str(PYTHON),"scripts/training/run_h_t01_h_t04_e300.py","--data-root",str(DATA_ROOT),"--gpu",GPU]
if RESUME: command.append("--resume")
log=(BATCH_DIR/"launcher.log").open("a",encoding="utf-8")
process=subprocess.Popen(command,cwd=ROOT,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
(BATCH_DIR/"launcher.pid").write_text(str(process.pid),encoding="utf-8")
print("PID:",process.pid)
print("log:",BATCH_DIR/"launcher.log")


## 3. 실시간 진행률

기존 지표에 Full Macro Dice, Top-3 epoch, 보존 checkpoint 수를 함께 표시합니다.


In [ ]:
PLANS={
"H-T01-E300":"ICHPlansTuneA04B06SgdE300V1",
"H-T02-E300":"ICHPlansTuneA02B08SgdE300V1",
"H-T03-E300":"ICHPlansTuneA04B06AdamWE300V1",
"H-T04-E300":"ICHPlansTuneA02B08AdamWE300V1",
}
PROGRESS_FILES={name:ROOT/f"runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__{plan}__3d_fullres/fold_0/progress.json" for name,plan in PLANS.items()}
def read_json(path):
    try:return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError,json.JSONDecodeError):return {}
def fmt(seconds):
    if seconds is None:return "-"
    seconds=int(seconds);h,r=divmod(seconds,3600);m,s=divmod(r,60);return f"{h:02d}:{m:02d}:{s:02d}"
while True:
    clear_output(wait=True);batch=read_json(BATCH_STATUS);items=batch.get("experiments",{});rows=[]
    for name,path in PROGRESS_FILES.items():
        p=read_json(path);b=items.get(name,{})
        rows.append({"실험":name,"상태":p.get("status",b.get("status","waiting")),"Epoch":f"{p.get('current_epoch',0)}/{p.get('total_epochs','-')}","진행률(%)":p.get("progress_percent",0),"ETA":fmt(p.get("eta_seconds")),"Loss":p.get("train_loss"),"Dice":p.get("pseudo_dice"),"Recall":p.get("pseudo_recall"),"Best Composite":p.get("best_composite"),"GPU 최대(GB)":p.get("gpu_max_memory_gb"),"현재 Full Macro Dice":p.get("full_macro_dice"),"Best Full Macro Dice":p.get("best_full_macro_dice"),"Best Macro Epoch":p.get("best_full_macro_epoch"),"Top-3 Macro Epochs":p.get("top_full_macro_epochs",[]),"보존 Checkpoints":p.get("retained_epoch_checkpoints",0)})
    if batch.get("status")=="running":overall="running"
    elif all(row["상태"]=="completed" for row in rows):overall="completed"
    else:overall=batch.get("status","starting")
    print("전체:",overall,"현재:",batch.get("current") or "-");display(pd.DataFrame(rows))
    if overall in {"completed","failed"}:break
    time.sleep(10)


## 4. 기존 T01~T04와 E300 최고 Macro Dice 비교

순차 배치는 학습 후 Top-3 주변 정밀 검증까지 자동 실행합니다. 완료 후 실행하세요.


In [ ]:
rows=[]
parent_plans={"H-T01":"ICHPlansTuneA04B06SgdV1","H-T02":"ICHPlansTuneA02B08SgdV1","H-T03":"ICHPlansTuneA04B06AdamWV1","H-T04":"ICHPlansTuneA02B08AdamWV1"}
for name,plan in {**parent_plans,**PLANS}.items():
    fold=ROOT/f"runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__{plan}__3d_fullres/fold_0"
    best=read_json(fold/"best_full_macro.json")
    if not best:
        history=read_json(fold/"full_validation_history.json")
        if isinstance(history,list) and history:best=max(history,key=lambda x:float(x["macro_dice_positive"]))
    rows.append({"실험":name,"Best Epoch":best.get("epoch"),"Best Macro Dice":best.get("macro_dice_positive")})
comparison=pd.DataFrame(rows).sort_values("Best Macro Dice",ascending=False)
display(comparison)
comparison.to_csv(BATCH_DIR/"parent_vs_e300.csv",index=False)
